In [142]:
# Sem regras
from ultralytics import RTDETR
import cv2
import matplotlib.pyplot as plt

def testar_modelo_centros(img_path, weights_path, conf_threshold=0.1, iou=0.1):
    # 1. Carregar o modelo
    model = RTDETR(weights_path)

    # 2. Realizar a predição
    results = model.predict(source=img_path, conf=conf_threshold, device='cuda')[0]

    # 3. Pegar a imagem original
    img_desenho = results.orig_img.copy()

    # 4. Extrair e ORDENAR as caixas da esquerda para a direita (pelo x1)
    boxes = results.boxes.xyxy.cpu().numpy()
    boxes = boxes[boxes[:, 0].argsort()]
    
    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = map(int, box) # Convertendo para int para o OpenCV
        
        # cv2.rectangle(img_desenho, (x1, y1), (x2, y2), (255, 100, 0), 2)
        
        # Calcular o centro
        center_x = int((x1 + x2) / 2)
        center_y = int((y1 + y2) / 2)
        
        # 5. Desenhar a bola vermelha (tamanho 12)
        cv2.circle(img_desenho, (center_x, center_y), 12, (0, 0, 255), -1)
        
        # 6. Escrever o número DENTRO da bola
        texto = str(i + 1)
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 0.4
        thickness = 1
        
        text_size = cv2.getTextSize(texto, font, font_scale, thickness)[0]
        text_x = center_x - text_size[0] // 2
        text_y = center_y + text_size[1] // 2
        
        # Print para conferência
        print(f"Biscoito {texto}: Caixa[{x1}, {y1}, {x2}, {y2}] Centro[{center_x}, {center_y}]")
        cv2.putText(img_desenho, texto, (text_x, text_y), font, 
                    font_scale, (255, 255, 255), thickness, cv2.LINE_AA)

    # 7. Converter e exibir
    img_rgb = cv2.cvtColor(img_desenho, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(15, 10))
    plt.imshow(img_rgb)
    plt.title(f"Detecção Completa: {len(boxes)} Biscoitos", fontsize=16)
    plt.axis('off')
    plt.show()

    return len(boxes)

In [143]:
# --- Exemplo de Uso ---
caminho_pesos = r"D:\Arquivos\ProjetosPython\PICOS\data\inputs\ia_models\RTDETR\best_20260105.pt"
caminho_imagem = r"D:\Arquivos\ProjetosPython\PICOS\data\outputs\teste_20260106\SM\SM_14_direito__20260106_011416.jpg"

total = testar_modelo_centros(caminho_imagem, caminho_pesos)

RuntimeError: CUDA error: unknown error
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [141]:
model_path = r"D:\Arquivos\ProjetosPython\PICOS\data\inputs\ia_models\RTDETR\best_20260105.pt"
torch_device = 'cuda'
model = RTDETR(model_path)
model.to(torch_device)

model.eval()

frame = r"D:\Arquivos\ProjetosPython\PICOS\data\outputs\teste_20260106\SM\SM_14_direito__20260106_011416.jpg"
results = model.predict(source=frame, conf=0.1, device='cuda')
detections = results[0].boxes  # Obter as detecções

# Extrair boxes e scores em um único loop
detections_info = [(detection.xyxy[0].tolist(), float(detection.conf)) for detection in detections]

boxes, scores = (zip(*detections_info) if detections_info else ([], []))  # Separar os boxes e scores em variáveis distintas

detections = []  # Cria uma lista para armazenar as detecções

# Itera sobre as caixas e pontuações para criar pares (box, score)
for box, score in zip(boxes, scores):
    detections.append([box, score])  # Não é necessário usar box.tolist() aqui

# Ordena as detecções pelo valor de x_min
detections_sorted = sorted(detections, key=lambda det: det[0][0])  # Ordena pelo x_min

print(len(detections_sorted))


image 1/1 D:\Arquivos\ProjetosPython\PICOS\data\outputs\teste_20260106\SM\SM_14_direito__20260106_011416.jpg: 640x640 28 Biscoitos, 80.6ms
Speed: 1.6ms preprocess, 80.6ms inference, 0.9ms postprocess per image at shape (1, 3, 640, 640)
28


In [144]:
# Com regras
from ultralytics import RTDETR
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torchvision.transforms as transforms

def load_model(weights_path):
    model = RTDETR(weights_path)

def rules_detection(frame, detections_sorted, perc_top, perc_bottom, perc_median, min_score, limit_center):
    def median_calculate(detections_sorted, min_score, height, perc_median):
        all_centers = ([])  # Lista para armazenar as coordenadas y dos centros detectados
        for idx, detection in enumerate(detections_sorted):
            score = detection[1]   # Pontuação de confiança
            x_min, y_min, x_max, y_max = detection[0]  # Coordenadas da caixa - y cresce de cima para baixo

            # Verificar se a pontuação é maior que o limite e se está entre as linhas de contagem
            if (score > min_score and y_max > line_limit_top and y_min < line_limit_bottom):
                center_y = (y_min + y_max) // 2   # Calcular o centro da caixa de detecção
                all_centers.append(center_y)  # Adiciona y à lista de centros

        if all_centers:
            median_y = int(np.median(all_centers))  # Obtém a mediana
            line_bottom_median = int(median_y + int((height * perc_median) / 2))
            line_top_median = int(median_y - int((height * perc_median) / 2))
            
            cv2.line(
                frame, 
                (640, median_y), 
                (640 + 640, median_y), 
                (255, 0, 0), 
                2,
            )  # Desenhar a linha horizontal na moda

            return all_centers, median_y, line_top_median, line_bottom_median
        
        return None, None, None, None

    def regra_biscoito_saltado(valid_detections, no_valid_centers):
        """Tira possíveis duplicidades no biscoito saltado.
        1 - Pega a mediana da área
        2 - Verifica se algum box tem mais de um centro dentro dele
        3 - Caso tenha algum em 2, verifica se a área é maior que duas vezes a área mediana (possível biscoito saltado)
        4 - Caso 3 seja verdade, considera apenas um desses centros
        5 - Caso tudo acima seja verdade, também precisa ter pelo menos 50% de intersecção entre os boxes
        """
        final_detections = []
        valid_centers = []
        valid_boxes = []
    
        # 1. Primeiro, calculamos as áreas de todas as detecções candidatas
        areas = []
        for detection in valid_detections:
            x_min, y_min, x_max, y_max = detection[0]
            areas.append((x_max - x_min) * (y_max - y_min))

        if not areas:
            return [], [], []

        # 2. Encontramos a área mediana do frame atual
        area_mediana = np.median(areas)

        # 3. Filtragem com a nova regra de área
        for idx, detection in enumerate(valid_detections):
            x_min, y_min, x_max, y_max = detection[0]
            coords = detection[0]
            
            area_atual = (x_max - x_min) * (y_max - y_min)
            center_x = int((x_min + x_max) // 2)
            center_y = int((y_min + y_max) // 2)
                
            # is_too_large = area_atual > (area_mediana * 2.0) # Retirar por enquanto
            is_too_large = 1
            
            is_duplicate = False
            for f_det in final_detections:
                fx1, fy1, fx2, fy2 = f_det[0]
                
                # 1. Teste do Centro (sua regra anterior)
                is_inside = (fx1 <= center_x <= fx2 and fy1 <= center_y <= fy2)
                
                # 2. Teste de Intersecção (Área 0.5)
                # Calcula as coordenadas do retângulo de intersecção
                inter_x1 = max(x_min, fx1)
                inter_y1 = max(y_min, fy1)
                inter_x2 = min(x_max, fx2)
                inter_y2 = min(y_max, fy2)
                
                inter_w = max(0, inter_x2 - inter_x1)
                inter_h = max(0, inter_y2 - inter_y1)
                inter_area = inter_w * inter_h
                
                # Verifica se a intersecção ocupa mais de 50% da área de qualquer um dos dois boxes
                area_f = (fx2 - fx1) * (fy2 - fy1)
                overlap_ratio = inter_area / min(area_atual, area_f)
                
                if is_inside or overlap_ratio > 0.5:
                    is_duplicate = True
                    break
            
            if is_too_large:
                if not is_duplicate:
                    valid_centers.append((center_x, center_y))
                    final_detections.append(detection)
                else:
                    no_valid_centers.append((center_x, center_y))
            else:
                if not is_duplicate:
                    valid_centers.append((center_x, center_y))
                    final_detections.append(detection)
                else:
                    no_valid_centers.append((center_x, center_y))

        return final_detections, valid_centers, no_valid_centers

    def filtrar_deteccoes(detections_sorted, centers, median_y, line_limit_top, line_limit_bottom, 
                      line_top_median, line_bottom_median, min_score, limit_center):
        """Filtra as detecções válidas com base em múltiplas condições."""
        valid_detections = []
        valid_centers = []
        no_valid_centers = []

        if centers and line_limit_bottom > median_y > line_limit_top:
            total_detectados = len(detections_sorted)
            for idx, detection in enumerate(detections_sorted):
                score = detection[1]
                x_min, y_min, x_max, y_max = detection[0]

                center_x = int((x_min + x_max) // 2)
                center_y = int((y_min + y_max) // 2)
                area = (x_max - x_min) * (y_max - y_min)

                if idx == 0 or idx == (total_detectados - 1):
                    min_score_din = 0.1
                else:
                    min_score_din = min_score

                test_score = score > min_score_din
                test_center = not any(np.linalg.norm(np.array([center_x, center_y]) - np.array(center)) < limit_center for center in valid_centers)
                test_median = (y_max > line_top_median and y_min < line_bottom_median)
                test_area = area >= 0  # Area precisa ser maior que x (excluir pedacos soltos)

                if test_score and test_median and test_center and test_area:
                        valid_detections.append(detection)
                        valid_centers.append((center_x, center_y))

                else:
                    no_valid_centers.append((center_x, center_y))

        valid_detections, valid_centers, no_valid_centers = regra_biscoito_saltado(valid_detections, no_valid_centers)

        return valid_detections, valid_centers, no_valid_centers
    
    def marcar_deteccoes(frame, valid_detections, limit_center, cor=(0, 0, 255)):
        """Desenha as detecções válidas (Box e Centro) no frame e atualiza o total."""
        total_detections = 0 
        
        for detection in valid_detections:
            # Extrair dados da detecção
            # d[0] são as coordenadas [x_min, y_min, x_max, y_max]
            coords = detection[0]
            x_min, y_min, x_max, y_max = map(int, coords)
            
            # Calcular centro para o desenho do círculo e texto
            center_x = (x_min + x_max) // 2
            center_y = (y_min + y_max) // 2
            
            total_detections += 1

            # 1. Desenhar a Bounding Box (Retângulo)
            # Usamos uma espessura fina (1 ou 2) para não poluir a imagem
            # cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), cor, 2)

            # 2. Desenhar o Centro (Círculo preenchido e borda)
            cv2.circle(frame, (center_x, center_y), limit_center - 1, cor, -1)
            cv2.circle(frame, (center_x, center_y), limit_center, (255, 0, 0), 1)

            # 3. Desenhar o número do índice (ID visual)
            cv2.putText(frame, str(total_detections), (center_x - 6, center_y + 3),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.35, (255, 255, 255), 1)
            
            # Opcional: Mostrar o Score da detecção acima da box
            # score = detection[1]
            # cv2.putText(frame, f'{score:.2f}', (x_min, y_min - 5), 
            #             cv2.FONT_HERSHEY_SIMPLEX, 0.3, cor, 1)

        return total_detections

    height, width = frame.shape[:2]

    # Define as posições das linhas
    line_limit_top = int(height * perc_top)    # Só conta quando a mediana entrar nesse range
    line_limit_bottom = int(height * perc_bottom)   # Só conta quando a mediana entrar nesse range
    dif_limit = line_limit_bottom - line_limit_top

    line_top_median = None  # É calculada a mediana das detecções, e só são contados biscoitos que estão naquela mediana + - um valor de range
    line_bottom_median = None   # É calculada a mediana das detecções, e só são contados biscoitos que estão naquela mediana + - um valor de range

    ### CALCULO DA MEDIANA
    all_centers, median_y, line_top_median, line_bottom_median = median_calculate(detections_sorted, min_score, height, perc_median)

    if all_centers and median_y is not None and line_limit_bottom > median_y > line_limit_top:
        ### FILTRAGEM
        valid_detections, valid_centers, no_valid_centers = filtrar_deteccoes(detections_sorted, all_centers, median_y, line_limit_top, line_limit_bottom, line_top_median, line_bottom_median, min_score, limit_center)
        ### MARCACAO
        total_detections = marcar_deteccoes(frame, valid_detections, limit_center)
        # _ = marcar_deteccoes(frame, no_valid_centers, limit_center, cor = (0, 0, 0))

        cv2.rectangle(frame, (0, 0), (frame.shape[1], 40), (80, 43, 30), -1)

        cv2.line(frame, (0, line_top_median), (frame.shape[1], line_top_median), (255, 0, 0), 2)
        cv2.line(frame,(0, line_bottom_median), (frame.shape[1], line_bottom_median), (255, 0, 0), 2)

        text_position = (0, 0)
        text = f'Total de Biscoitos: {total_detections}'
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 1
        thickness = 2
        (text_width, text_height), _ = cv2.getTextSize(text, font, font_scale, thickness)
        text_x = 10  # margem esquerda
        text_y = int((40 + text_height) / 2)  # centralizado verticalmente
        
        cv2.putText(frame, text, (text_x, text_y), font, font_scale, (255, 255, 255), thickness)

        return frame, total_detections

    else:
        return frame, 0   

def run_model(device, model, frame, threshold=0.5, limit_center=12, perc_top=0.1, perc_bottom=0.9, perc_median=0.3):
    # 1. Realizar a predição diretamente no frame (OpenCV BGR)
    results = model.predict(source=frame, conf=0.1, device=device)[0]

    # 2. Extrair dados brutos
    boxes_raw = results.boxes.xyxy.cpu().numpy()  # Coordenadas [x1, y1, x2, y2]
    scores_raw = results.boxes.conf.cpu().numpy()

    # 3. Calcular centros para ordenação da esquerda para a direita
    centers_x = (boxes_raw[:, 0] + boxes_raw[:, 2]) / 2
    indices_ordenados = np.argsort(centers_x)

    # 4. Montar a lista 'detections_sorted' exigida pela sua função rules_detection
    # Formato esperado: [([x1, y1, x2, y2], score), ...]
    detections_sorted = []
    for idx in indices_ordenados:
        detections_sorted.append((boxes_raw[idx], scores_raw[idx]))

    # 5. Chamar sua função de regras (ela já faz a filtragem, marcação e desenho)
    # Importante: Passe uma cópia do frame para não alterar o original se precisar dele depois
    frame_processado, total_detections = rules_detection(
        frame=frame.copy(),
        detections_sorted=detections_sorted,
        perc_top=perc_top,
        perc_bottom=perc_bottom,
        perc_median=perc_median,
        min_score=threshold,
        limit_center=limit_center
    )

    # 6. Exibir o resultado usando Matplotlib (opcional, conforme seu código original)
    img_rgb = cv2.cvtColor(frame_processado, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(15, 10))
    plt.imshow(img_rgb)
    plt.title(f"Resultado Final: {total_detections} Biscoitos", fontsize=16)
    plt.axis('off')
    plt.show()

    return total_detections

In [145]:

modelo_treinado = RTDETR(r"D:\Arquivos\ProjetosPython\PICOS\data\inputs\ia_models\RTDETR\best_20260105.pt")
imagem_array = cv2.imread(r"D:\Arquivos\ProjetosPython\PICOS\data\outputs\teste_20260106\SM\SM_14_direito__20260106_011416.jpg")

total = run_model(
    device='cuda', 
    model=modelo_treinado, 
    frame=imagem_array,
    threshold=0.1,
    limit_center=12
)
print(f"Contagem final na esteira: {total}")



RuntimeError: CUDA error: unknown error
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
